# Experiment 13 — Random-forest mixture with a neural gate

Three random-forest experts see the same compact feature set but use different depth, leaf-size, and feature-sampling settings. A neural soft gate learns non-negative, per-row weights over their predictions.

The validation period is never used to fit a forest, preprocessor, category vocabulary, or gate. The gate is trained only on chronological out-of-fold (OOF) forest predictions from the fit period.

**Status:** the notebook defines the experts and leakage-safe gating workflow, but its final gate-training and holdout-evaluation cells have no saved execution outputs. No validation RMSE is therefore reported, and this prepared experiment is excluded from numerical comparisons until those cells are run.

In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents]
project_root = next((root for root in roots if (root / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
sys.path.insert(0, str(project_root / 'exploration' / 'metul'))
from preprocessing import RentalPreprocessor, load_datasets

TARGET = 'MONTHLY_RENT'
N_SPLITS = 3
RF_TREES = int(os.environ.get('RF_GATE_TREES', '150'))
GATE_EPOCHS = int(os.environ.get('RF_GATE_EPOCHS', '150'))
BATCH_SIZE = 1024
np.random.seed(42)
torch.manual_seed(42)

compact_features = [
    'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM',
    'approval_date_num', 'approval_month', 'coe_quota_premium_mean',
    'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', 'mall_dist_nearest_km',
]
EXPERTS = {
    'rf_regularized': dict(max_depth=18, min_samples_leaf=8, max_features=0.65, random_state=11),
    'rf_balanced': dict(max_depth=None, min_samples_leaf=3, max_features=0.80, random_state=22),
    'rf_local': dict(max_depth=None, min_samples_leaf=1, max_features=0.45, random_state=33),
}


In [2]:
_, datasets = load_datasets(project_root / 'data')
raw = datasets['train'].copy()
raw['period'] = pd.to_datetime(raw['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
periods = np.sort(raw['period'].unique())
cutoff = periods[int(len(periods) * 0.80)]
raw_fit = raw.loc[raw['period'] < cutoff].copy()
raw_valid = raw.loc[raw['period'] >= cutoff].copy()
print(f'Cutoff={cutoff}; fit={len(raw_fit):,}; validation={len(raw_valid):,}; trees/expert={RF_TREES}')

def transform_fit_valid(fit_rows, predict_rows):
    processor = RentalPreprocessor(datasets)
    fit_base = processor.fit_transform(fit_rows.drop(columns=['period', TARGET], errors='ignore'))
    predict_base = processor.transform(predict_rows.drop(columns=['period', TARGET], errors='ignore'))
    return fit_base[compact_features].reset_index(drop=True), predict_base[compact_features].reset_index(drop=True)

fit_x, valid_x = transform_fit_valid(raw_fit, raw_valid)
fit_y = raw_fit[TARGET].astype(float).reset_index(drop=True)
valid_y = raw_valid[TARGET].astype(float).reset_index(drop=True)


Cutoff=2024-05; fit=117,988; validation=32,012; trees/expert=150
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whites

In [3]:
categorical_columns = fit_x.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
numeric_columns = [column for column in compact_features if column not in categorical_columns]

def make_expert(config):
    encoder = ColumnTransformer([
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median'))]), numeric_columns),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_columns),
    ])
    forest = RandomForestRegressor(n_estimators=RF_TREES, n_jobs=-1, **config)
    return Pipeline([('encode', encoder), ('forest', forest)])

def metric(name, actual, prediction):
    return {
        'model': name,
        'RMSE': mean_squared_error(actual, prediction) ** 0.5,
        'MAE': mean_absolute_error(actual, prediction),
    }

# OOF rows are the only data the gate is allowed to learn from. Each fold's
# forests and preprocessor are fit strictly before that fold's periods.
fit_periods = np.sort(raw_fit['period'].unique())
oof_parts = []
for fold, (train_period_index, oof_period_index) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(fit_periods), start=1):
    early_periods = fit_periods[train_period_index]
    later_periods = fit_periods[oof_period_index]
    early_raw = raw_fit.loc[raw_fit['period'].isin(early_periods)].copy()
    later_raw = raw_fit.loc[raw_fit['period'].isin(later_periods)].copy()
    early_x, later_x = transform_fit_valid(early_raw, later_raw)
    part = later_x[['TOWN', 'BLOCK', 'FLAT_TYPE', 'approval_date_num']].copy()
    part['target'] = later_raw[TARGET].to_numpy()
    part['fold'] = fold
    for name, config in EXPERTS.items():
        part[name] = make_expert(config).fit(early_x, early_raw[TARGET]).predict(later_x)
    oof_parts.append(part)
    print(f'OOF fold {fold}: train={len(early_raw):,}; OOF={len(later_raw):,}')

oof = pd.concat(oof_parts, ignore_index=True)
assert oof[list(EXPERTS)].notna().all().all()


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


KeyboardInterrupt: 

In [ ]:
# Keep the final OOF fold as a gate early-stopping period; it is not the
# outer validation period used for the reported experiment result.
gate_stop_fold = oof['fold'].max()
gate_train = oof.loc[oof['fold'] < gate_stop_fold].copy()
gate_stop = oof.loc[oof['fold'] == gate_stop_fold].copy()

def make_vocab(values):
    return {value: index + 1 for index, value in enumerate(sorted(values.dropna().astype(str).unique()))}

vocabs = {column: make_vocab(gate_train[column]) for column in ['TOWN', 'BLOCK', 'FLAT_TYPE']}
prediction_mean = gate_train[list(EXPERTS)].mean()
prediction_std = gate_train[list(EXPERTS)].std().replace(0, 1)
date_mean = gate_train['approval_date_num'].mean()
date_std = gate_train['approval_date_num'].std() or 1.0

def gate_inputs(frame):
    categories = [
        torch.tensor(frame[column].astype(str).map(vocabs[column]).fillna(0).astype(int).to_numpy(), dtype=torch.long)
        for column in ['TOWN', 'BLOCK', 'FLAT_TYPE']
    ]
    date = torch.tensor(((frame['approval_date_num'] - date_mean) / date_std).to_numpy(), dtype=torch.float32).reshape(-1, 1)
    scaled_predictions = torch.tensor(((frame[list(EXPERTS)] - prediction_mean) / prediction_std).to_numpy(), dtype=torch.float32)
    raw_predictions = torch.tensor(frame[list(EXPERTS)].to_numpy(), dtype=torch.float32)
    return *categories, date, scaled_predictions, raw_predictions

class RandomForestGate(nn.Module):
    def __init__(self):
        super().__init__()
        self.town = nn.Embedding(len(vocabs['TOWN']) + 1, 8)
        self.block = nn.Embedding(len(vocabs['BLOCK']) + 1, 16)
        self.flat_type = nn.Embedding(len(vocabs['FLAT_TYPE']) + 1, 4)
        self.network = nn.Sequential(
            nn.Linear(8 + 16 + 4 + 1 + len(EXPERTS), 48),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(48, len(EXPERTS)),
        )

    def forward(self, town, block, flat_type, date, scaled_predictions, raw_predictions):
        features = torch.cat([self.town(town), self.block(block), self.flat_type(flat_type), date, scaled_predictions], dim=1)
        weights = torch.softmax(self.network(features), dim=1)
        return (weights * raw_predictions).sum(dim=1), weights

gate = RandomForestGate()
optimizer = torch.optim.AdamW(gate.parameters(), lr=0.001, weight_decay=0.001)
train_inputs = gate_inputs(gate_train)
train_target = torch.tensor(gate_train['target'].to_numpy(), dtype=torch.float32)
loader = DataLoader(TensorDataset(*train_inputs, train_target), batch_size=BATCH_SIZE, shuffle=True)
best_state, best_loss, stale = None, np.inf, 0
for epoch in range(GATE_EPOCHS):
    gate.train()
    for town, block, flat_type, date, scaled, raw_prediction, target in loader:
        prediction, _ = gate(town, block, flat_type, date, scaled, raw_prediction)
        loss = ((prediction - target) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    gate.eval()
    with torch.no_grad():
        stop_prediction, _ = gate(*gate_inputs(gate_stop))
        stop_loss = ((stop_prediction - torch.tensor(gate_stop['target'].to_numpy(), dtype=torch.float32)) ** 2).mean().item()
    if stop_loss < best_loss:
        best_loss = stop_loss
        best_state = {key: value.detach().clone() for key, value in gate.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= 20:
            break
gate.load_state_dict(best_state)
print(f'Gate stopped after {epoch + 1} epochs; OOF stop RMSE={best_loss ** 0.5:.3f}')


In [ ]:
# Refit each forest on all outer-fit rows, then predict the untouched outer validation period.
validation_predictions = valid_x[['TOWN', 'BLOCK', 'FLAT_TYPE', 'approval_date_num']].copy()
for name, config in EXPERTS.items():
    validation_predictions[name] = make_expert(config).fit(fit_x, fit_y).predict(valid_x)

gate.eval()
with torch.no_grad():
    gated_prediction, validation_weights = gate(*gate_inputs(validation_predictions))
validation_predictions['neural_gate'] = gated_prediction.numpy()
validation_predictions['equal_blend'] = validation_predictions[list(EXPERTS)].mean(axis=1)

comparison = pd.DataFrame([
    metric(name, valid_y, validation_predictions[name]) for name in EXPERTS
] + [
    metric('equal blend', valid_y, validation_predictions['equal_blend']),
    metric('neural soft gate', valid_y, validation_predictions['neural_gate']),
])
display(comparison.sort_values('RMSE'))

weight_summary = pd.DataFrame(validation_weights.numpy(), columns=list(EXPERTS)).agg(['mean', 'std', 'min', 'max']).T
display(weight_summary)

# Larger mean weight means the gate selected that expert more often on validation.
# Compare gate RMSE with the best individual forest and equal blend before using it.
